# Chapter 2 · 수익률 ARIMA·변동성 GARCH

**연구질문:** 수익률 평균은 ARIMA로 예측 가능한가? 변동성은 GARCH(1,1)가 단순 이동표준편차보다 잘 예측하는가?

> 위에서 아래로 실행하면 끝까지 동작합니다. 온라인 실패 시 합성 대체(`DATA_KIND` 표시).

## 1. 핵심 개념·수식

**ARIMA$(p,d,q)$:** $\phi(L)(1-L)^d y_t = c + \theta(L)\varepsilon_t$, $\ \varepsilon_t \sim WN(0,\sigma^2)$.

**SARIMA$(p,d,q)(P,D,Q)_m$:** 계절 차분 $(1-L^m)^D$와 계절 AR/MA 다항식 $\Phi(L^m),\Theta(L^m)$을 추가한다.

**차수 선택:** $\mathrm{AIC}=2k-2\ln\hat{L}$. AIC는 **적합 내 기준**이며 외표본 성능을 보장하지 않는다. 항상 시간순 홀드아웃에서 기준선과 비교한다.

**예측 오차 지표:** $\mathrm{MAE}$, $\mathrm{RMSE}$, $\mathrm{MASE}=\dfrac{\mathrm{MAE}}{\frac{1}{T-1}\sum_{t=2}^{T}|y_t-y_{t-1}|}$ (분모는 학습구간 1단계 나이브의 평균절대오차; 1보다 작으면 학습구간 나이브보다 낫다).

**잔차 진단:** 잔차가 백색잡음이면 Ljung–Box $Q$ 검정에서 기각되지 않아야 한다.

**GARCH(1,1):** $r_t=\mu+\varepsilon_t,\ \varepsilon_t=\sigma_t z_t,\ \sigma_t^2=\omega+\alpha\varepsilon_{t-1}^2+\beta\sigma_{t-1}^2$, 정상성 조건 $\alpha+\beta<1$. 외부 패키지 없이 최대우도로 직접 추정한다.

In [ ]:
import os, io, json, subprocess, sys, warnings, urllib.request, urllib.parse
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
# 네트워크 없이 합성자료로만 실행하려면 환경변수 TSLAB_OFFLINE=1 로 실행하거나 아래를 False로 바꾼다.
USE_LIVE_DATA = os.environ.get("TSLAB_OFFLINE") != "1"
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True,
                     "axes.spines.top": False, "axes.spines.right": False})

def ensure(pkg, mod=None):
    # 없으면 설치를 시도한다. 실패해도 노트북은 계속 실행되고 해당 선택 기능만 건너뛴다.
    try:
        __import__(mod or pkg); return True
    except Exception:
        if not USE_LIVE_DATA:
            return False
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True, timeout=300)
            __import__(mod or pkg); return True
        except Exception as e:
            print(f"[건너뜀] {pkg} 설치/가져오기 실패: {type(e).__name__}")
            return False

def fetch_bytes(url, timeout=60):
    req = urllib.request.Request(url, headers={"User-Agent": "ts-lab-course/1.0 (education)"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

# ---- 평가 지표와 기준선 ----
def mae(a, f):  return float(np.mean(np.abs(np.asarray(a, float) - np.asarray(f, float))))
def rmse(a, f): return float(np.sqrt(np.mean((np.asarray(a, float) - np.asarray(f, float)) ** 2)))
def mase(a, f, train, m=1):
    train = np.asarray(train, float)
    scale = np.mean(np.abs(train[m:] - train[:-m]))
    return mae(a, f) / scale
def naive_fc(train, h):  return np.repeat(np.asarray(train, float)[-1], h)
def snaive_fc(train, h, m):
    train = np.asarray(train, float)
    return np.tile(train[-m:], int(np.ceil(h / m)))[:h]
def drift_fc(train, h):
    train = np.asarray(train, float)
    slope = (train[-1] - train[0]) / (len(train) - 1)
    return train[-1] + slope * np.arange(1, h + 1)
def show(df, digits=4):
    print(df.round(digits).to_string())

In [ ]:
def garch_sim(n, seed, omega=0.02, alpha=0.08, beta=0.90):
    rng = np.random.default_rng(seed); r = np.zeros(n); s2 = omega / (1 - alpha - beta)
    for t in range(n):
        r[t] = np.sqrt(s2) * rng.standard_normal()
        s2 = omega + alpha * r[t] ** 2 + beta * s2
    return r + 0.04                                            # 일수익률(%) , 약간의 양의 평균

def load_finance():
    px, kind = None, "SYNTHETIC (NOT real data): GARCH(1,1) 수익률로 만든 가격"
    if USE_LIVE_DATA and ensure("yfinance"):
        try:
            import yfinance as yf
            raw = yf.download("SPY", start="2015-01-01", progress=False, auto_adjust=True)
            c = raw["Close"]
            if isinstance(c, pd.DataFrame):
                c = c.iloc[:, 0]
            c = c.dropna().astype(float)
            if len(c) > 500:
                px, kind = c, "REAL: SPY (Yahoo Finance, yfinance)"
        except Exception as e:
            print("온라인 수집 실패 -> 합성 대체:", type(e).__name__, str(e)[:120])
    if px is None:
        r = garch_sim(2500, SEED)
        px = pd.Series(100 * np.exp(np.cumsum(r / 100)), index=pd.bdate_range("2015-01-01", periods=2500))
    px.index = pd.to_datetime(px.index)
    if getattr(px.index, "tz", None) is not None:
        px.index = px.index.tz_localize(None)
    return dict(y=px.rename("price"), kind=kind, m=5, freq="B", label="SPY 종가", unit="USD")

D = load_finance()
y, DATA_KIND, M, FREQ = D["y"], D["kind"], D["m"], D["freq"]
ret = (np.log(y).diff().dropna() * 100).rename("log_return_pct")      # 로그수익률(%)
print("DATA_KIND:", DATA_KIND); print("관측 수:", len(y), "| 기간:", y.index.min().date(), "~", y.index.max().date())

## 2. 분할과 수익률 평균 예측 (ARIMA)

In [ ]:
H = 250                                    # 마지막 약 1년을 검증으로 사용
r_all = ret.astype(float)
r_tr, r_te = r_all.iloc[:-H].to_numpy(), r_all.iloc[-H:].to_numpy()
print(f"학습 {len(r_tr)} | 검증 {len(r_te)} (로그수익률 %)")

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
fit = ARIMA(r_tr, order=(1, 0, 1)).fit()
print("ARIMA(1,0,1) 추정 완료 | 잔차 Ljung-Box(10) p =",
      round(float(acorr_ljungbox(fit.resid, lags=[10], return_df=True)["lb_pvalue"].iloc[0]), 4))
# 검증구간 다기간 예측은 평균으로 수렴하므로, 일일 1단계 예측을 갱신 없이 비교하기보다 기준선(역사적 평균)과 비교한다.
arima_fc = np.asarray(fit.get_forecast(H).predicted_mean)
hist_mean = np.full(H, r_tr.mean()); zero = np.zeros(H)
def dm_test(e1, e2, h=1):
    # Diebold-Mariano (제곱오차 손실, Newey-West 분산, 단순 버전)
    d = np.asarray(e1) ** 2 - np.asarray(e2) ** 2; n = len(d); dbar = d.mean()
    L = max(h - 1, 0); g0 = np.var(d, ddof=0); var = g0
    for l in range(1, L + 1):
        var += 2 * (1 - l / (L + 1)) * np.mean((d[l:] - dbar) * (d[:-l] - dbar))
    stat = dbar / np.sqrt(var / n); return stat, 2 * (1 - stats.norm.cdf(abs(stat)))
sse = lambda f: float(np.sum((r_te - f) ** 2))
r2_oos = 1 - sse(arima_fc) / sse(hist_mean)             # 역사적 평균 대비 표본 외 R²
print(f"RMSE: ARIMA={rmse(r_te, arima_fc):.4f}  역사평균={rmse(r_te, hist_mean):.4f}  0예측={rmse(r_te, zero):.4f}")
print(f"표본 외 R²(역사평균 대비) = {r2_oos:.4f}  (0 이하이면 평균보다 낫지 않음)")
st, pv = dm_test(r_te - arima_fc, r_te - hist_mean)
print(f"Diebold-Mariano(ARIMA vs 역사평균): stat={st:.3f}, p={pv:.3f}")

## 3. 변동성 예측 (GARCH)

In [ ]:
# GARCH(1,1) 최대우도 추정 (%, 단위 변환으로 수치 안정성 확보)
def garch11_nll(theta, r):
    mu, omega, alpha, beta = theta
    e = r - mu; n = len(r); s2 = np.empty(n); s2[0] = np.var(r)
    for t in range(1, n):
        s2[t] = omega + alpha * e[t - 1] ** 2 + beta * s2[t - 1]
    return 0.5 * np.sum(np.log(2 * np.pi * s2) + e ** 2 / s2)

def fit_garch11(r):
    x0 = [r.mean(), 0.05 * np.var(r), 0.08, 0.88]
    cons = [{"type": "ineq", "fun": lambda th: 0.999 - th[2] - th[3]}]
    bnds = [(-1, 1), (1e-8, None), (1e-6, 0.5), (1e-6, 0.999)]
    res = optimize.minimize(garch11_nll, x0, args=(r,), method="SLSQP", bounds=bnds, constraints=cons,
                            options={"maxiter": 200, "ftol": 1e-9})
    return res

def garch_filter(theta, r):
    mu, omega, alpha, beta = theta; e = r - mu; s2 = np.empty(len(r)); s2[0] = np.var(r)
    for t in range(1, len(r)):
        s2[t] = omega + alpha * e[t - 1] ** 2 + beta * s2[t - 1]
    return s2, e

g = fit_garch11(r_tr)
mu_, om_, al_, be_ = g.x
print(f"GARCH(1,1): mu={mu_:.4f} omega={om_:.5f} alpha={al_:.3f} beta={be_:.3f} | alpha+beta={al_+be_:.3f} | 수렴={g.success}")
# 검증구간 1단계 조건부 분산 예측(매일 갱신)과 기준선(최근 21일 표본분산) 비교, 실현변동성 대용치로 e_t^2 사용
r_full = r_all.to_numpy(); s2_full, e_full = garch_filter(g.x, r_full)
n_tr = len(r_tr)
garch_var = om_ + al_ * e_full[n_tr - 1:-1] ** 2 + be_ * s2_full[n_tr - 1:-1]       # t의 분산을 t-1 정보로 예측
roll_var = np.array([np.var(r_full[t - 21:t]) for t in range(n_tr, len(r_full))])
proxy = (r_te - mu_) ** 2
def qlike(proxy, f): return float(np.mean(np.log(f) + proxy / f))
print(f"QLIKE(작을수록 좋음): GARCH={qlike(proxy, garch_var):.4f}  21일 롤링분산={qlike(proxy, roll_var):.4f}")
print(f"MSE(분산 대용치): GARCH={np.mean((proxy-garch_var)**2):.4f}  롤링={np.mean((proxy-roll_var)**2):.4f}")
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(r_all.index[-H:], np.sqrt(proxy), alpha=0.3, label="|e_t| (실현 대용치)")
ax.plot(r_all.index[-H:], np.sqrt(garch_var), label="GARCH sigma")
ax.plot(r_all.index[-H:], np.sqrt(roll_var), label="21일 롤링 sigma"); ax.legend(); plt.tight_layout(); plt.show()

## 3. 결과 해석 작성
1. 수익률 평균 예측: 표본 외 R²가 0 이하이고 DM 검정이 기각되지 않으면 "평균 예측은 어렵다"가 합리적 결론이다. 효율적 시장 가설과 연결하되, 단일 구간 결과임을 명시한다.
2. 변동성 예측: QLIKE와 MSE 모두에서 GARCH가 롤링분산보다 낮은지 확인하고, 실현변동성 대용치($e_t^2$)의 잡음을 한계로 적는다.
3. `DATA_KIND`가 합성이면 GARCH 모수는 생성 모수(약 α=0.08, β=0.90)와 비교하는 **추정기 검증**이다. 실제 시장에 대한 증거가 아니다.
4. 거래비용·위험조정 성과 없이 경제적 유의성을 주장하지 않는다.